In [ ]:
import sys
sys.path.insert(0, '../../build-py')
import orderbook
import pandas as pd
import matplotlib.pyplot as plt

s = orderbook.load_replay('../../data/mock.NASDAQ_ITCH50')
print('orderbook module loaded:', orderbook.__doc__)

In [ ]:
# Step through 10,000 events, sampling book state every 100 steps
records = []
i = 0
while s.step() and i < 10_000:
    i += 1
    if i % 100 == 0:
        snap = s.snapshot()
        records.append({
            'event': i,
            'best_bid':  snap['best_bid']  / 10_000,
            'best_ask':  snap['best_ask']  / 10_000,
            'spread':   (snap['best_ask'] - snap['best_bid']) / 10_000,
            'order_count': snap['order_count'],
        })
df = pd.DataFrame(records)
df.head()

In [ ]:
# Bid/ask spread over time
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
df.plot(x='event', y=['best_bid', 'best_ask'], ax=axes[0],
        title='Top-of-Book Prices', ylabel='Price ($)')
df.plot(x='event', y='spread', ax=axes[1],
        title='Bid-Ask Spread', ylabel='Spread ($)', color='orange')
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display

# Depth ladder at end of replay
bid_levels = s.query_top_n(10, orderbook.Side.BID)
ask_levels = s.query_top_n(10, orderbook.Side.ASK)
depth_df = pd.DataFrame({
    'bid_price':  [p/10_000 for p,_ in bid_levels],
    'bid_volume': [v        for _,v in bid_levels],
    'ask_price':  [p/10_000 for p,_ in ask_levels],
    'ask_volume': [v        for _,v in ask_levels],
})
display(depth_df)

# Final snapshot
snap = s.snapshot()
print(f"Best bid:     ${snap['best_bid']/10_000:.4f}  (volume: {snap['best_bid_volume']:,})")
print(f"Best ask:     ${snap['best_ask']/10_000:.4f}  (volume: {snap['best_ask_volume']:,})")
print(f"Order count:  {snap['order_count']:,}")
print(f"Checksum:     {snap['checksum']}")